策略1：重要性评分

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv('SILICON_API_KEY')

llm = init_chat_model(
    "Qwen/Qwen3-8B",
    model_provider='openai',
    base_url="https://api.siliconflow.cn/v1",
    api_key=API_KEY,
    temperature=0.0
)


def calculate_importance(fact: str) -> float:
    """评估事实的重要性 (0.0 - 1.0)"""
    prompt = f"""
        评估以下事实的重要性（0.0 到 1.0）：
        {fact}

        重要性评分（只返回数字）：
    """
    response = llm.invoke([{"role": "user", "content": prompt}])
    try:
        return float(response.content.strip())
    except:
        return 0.5

def prune_memories(memories: list[tuple[str, float]], max_count: int = 50):
    """保留最重要的记忆"""
    # 按重要性排序
    sorted_memories = sorted(memories, key=lambda x: x[1], reverse=True)
    # 只保留前 max_count 个
    return sorted_memories[:max_count]

策略2：时间衰减

In [ ]:
from datetime import datetime
import math

def time_decay_weight(created_at: datetime, half_life_days: int = 30) -> float:
    """计算时间衰减权重"""
    days_ago = (datetime.now() - created_at).days
    return math.exp(-days_ago * math.log(2) / half_life_days)


def retrieve_with_decay(memories: list, query: str):
    """检索时考虑时间衰减"""
    scored_memories = []
    for mem in memories:
        relevance_score = calculate_relevance(mem["content"], query)
        time_weight = time_decay_weight(mem["created_at"])
        final_score = relevance_score * time_weight
        scored_memories.append((mem, final_score))

    # 返回得分最高的
    scored_memories.sort(key=lambda x: x[1], reverse=True)
    return [mem for mem, score in scored_memories[:5]]